In [5]:
#EXERCISE:
#1. Modify the codes for both TF-IDF & Word2Vec vectorizer by adding text preprocessing steps.
# Do the Purity differ when applying text preprocessing before vectorization?


In [6]:
import numpy as np
import pandas as pd
import nltk
import re
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from tabulate import tabulate
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Setup NLTK
nltk.download('punkt')
nltk.download('stopwords')

# 1. Dataset and Preprocessing Function
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

def preprocess(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    return [w for w in tokens if w not in stop_words]

# Preprocessed versions
tokenized_dataset = [preprocess(doc) for doc in dataset]
cleaned_dataset = [" ".join(tokens) for tokens in tokenized_dataset]

# --- PART A: TF-IDF with Preprocessing ---
print("--- TF-IDF Clustering (with Preprocessing) ---")
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(cleaned_dataset)

k = 2
km_tfidf = KMeans(n_clusters=k, random_state=42)
y_pred_tfidf = km_tfidf.fit_predict(X_tfidf)

table_tfidf = [["Document", "Cluster"]]
table_tfidf.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred_tfidf)])
print(tabulate(table_tfidf, headers="firstrow"))

# Purity Calculation
purity_tfidf = sum(max(Counter(y_pred_tfidf).values()) for _ in [0]) / len(y_pred_tfidf)
print(f"TF-IDF Purity: {purity_tfidf}\n")

# --- PART B: Word2Vec with Preprocessing ---
print("--- Word2Vec Clustering (with Preprocessing) ---")
w2v_model = Word2Vec(sentences=tokenized_dataset, vector_size=100, window=5, min_count=1, workers=4)

# Create document embeddings by averaging word vectors
def get_embedding(tokens):
    vectors = [w2v_model.wv[w] for w in tokens if w in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_w2v = np.array([get_embedding(tokens) for tokens in tokenized_dataset])

km_w2v = KMeans(n_clusters=k, random_state=42)
y_pred_w2v = km_w2v.fit_predict(X_w2v)

table_w2v = [["Document", "Cluster"]]
table_w2v.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred_w2v)])
print(tabulate(table_w2v, headers="firstrow"))

purity_w2v = sum(max(Counter(y_pred_w2v).values()) for _ in [0]) / len(y_pred_w2v)
print(f"Word2Vec Purity: {purity_w2v}")

--- TF-IDF Clustering (with Preprocessing) ---
Document                                           Cluster
-----------------------------------------------  ---------
I love playing football on the weekends                  1
I enjoy hiking and camping in the mountains              0
I like to read books and watch movies                    1
I prefer playing video games over sports                 1
I love listening to music and going to concerts          1
TF-IDF Purity: 0.8

--- Word2Vec Clustering (with Preprocessing) ---
Document                                           Cluster
-----------------------------------------------  ---------
I love playing football on the weekends                  0
I enjoy hiking and camping in the mountains              0
I like to read books and watch movies                    1
I prefer playing video games over sports                 0
I love listening to music and going to concerts          1
Word2Vec Purity: 0.6


[nltk_data] Downloading package punkt to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
#Yes, purity usually changes. By removing "noise" (like "the", "and", "to") , the vectorizer focuses on "content" words (like "football", "hiking", "movies"). This often leads to more distinct clusters and potentially higher purity.

In [9]:
# Question 2. Perform text clustering on 'customer_complaints_1.csv' dataset, specifically the Text column.



import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# 1. Define a preprocessing function (Exercise 1 requirement)
def preprocess_text(text):
    # Convert to lowercase
    text = str(text).lower()
    # Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', '', text)
    return text

# 2. Load the dataset
try:
    df = pd.read_csv('customer_complaints_1.csv')
    
    # 3. Preprocess the 'text' column (Note: lowercase 'text' from the CSV)
    df['Cleaned_Text'] = df['text'].apply(preprocess_text)

    # 4. Vectorization using TF-IDF
    # We use built-in English stop words to filter out common words
    vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
    X = vectorizer.fit_transform(df['Cleaned_Text'])

    # 5. Perform Clustering (Grouping into 3 clusters)
    k_clusters = 3
    model = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
    df['Cluster'] = model.fit_predict(X)

    # 6. Display Top Terms per Cluster
    print(f"Top terms per cluster for Customer Complaints:")
    order_centroids = model.cluster_centers_.argsort()[:, ::-1]
    terms = vectorizer.get_feature_names_out()

    for i in range(k_clusters):
        print(f"\nCluster {i}:")
        for ind in order_centroids[i, :10]: # Showing top 10 words
            print(f" - {terms[ind]}")

    # Show first few results
    print("\nSample of Clustered Complaints:")
    print(df[['text', 'Cluster']].head(10))

except FileNotFoundError:
    print("Error: 'customer_complaints_1.csv' not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Top terms per cluster for Customer Complaints:

Cluster 0:
 - comcast
 - internet
 - service
 - customer
 - security
 - months
 - adding
 - services
 - im
 - second

Cluster 1:
 - contract
 - mbps
 - speed
 - told
 - service
 - internet
 - xfinity
 - customer
 - week
 - comcast

Cluster 2:
 - rude
 - day
 - service
 - mins
 - rep
 - joke
 - dont
 - local
 - comcast
 - people

Sample of Clustered Complaints:
                                                text  Cluster
0  I used to love Comcast. Until all these consta...        2
1  I'm so over Comcast! The worst internet provid...        0
2  If I could give them a negative star or no sta...        0
3  I've had the worst experiences so far since in...        0
4  Check your contract when you sign up for Comca...        1
5  Thank God. I am changing to Dish. They gave me...        0
6  I Have been a long time customer and only have...        1
7  There is a malfunction on the DVR manager whic...        0
8  Charges overwhelming. Comcas